# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [2]:
import open3d as o3d, numpy as np, copy

# Helper (add once)
def draw_registrations(source, target, T=None, recolor=True):
    s, t = copy.deepcopy(source), copy.deepcopy(target)
    if recolor:
        s.paint_uniform_color([1, 0.706, 0])
        t.paint_uniform_color([0, 0.651, 0.929])
    if T is not None:
        s.transform(T)
    o3d.visualization.draw_geometries([s, t])

# 1) Load
source = o3d.io.read_point_cloud("ICP/r1.pcd")
target = o3d.io.read_point_cloud("ICP/r2.pcd")
assert len(source.points) and len(target.points), "Could not load ICP/r1.pcd or ICP/r2.pcd"

# 2) Downsample & normals
voxel_size = 0.05
src_d = source.voxel_down_sample(voxel_size);  tgt_d = target.voxel_down_sample(voxel_size)
src_d.estimate_normals();                      tgt_d.estimate_normals()

# 3) FPFH features
src_f = o3d.pipelines.registration.compute_fpfh_feature(
    src_d, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100))
tgt_f = o3d.pipelines.registration.compute_fpfh_feature(
    tgt_d, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100))

# 4) Global registration (RANSAC)
ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    src_d, tgt_d, src_f, tgt_f,
    True, voxel_size*1.5,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    4
)

# (Optional) refine with ICP point-to-plane on full clouds
source.estimate_normals(); target.estimate_normals()
icp = o3d.pipelines.registration.registration_icp(
    source, target, voxel_size, ransac.transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

# 5) Visualize (use RANSAC or ICP transform)
draw_registrations(source, target, icp.transformation, True)


In [3]:
import open3d as o3d

# 1) Load
car1 = o3d.io.read_point_cloud("global_registration/car1.ply")
car2 = o3d.io.read_point_cloud("global_registration/car2.ply")
assert len(car1.points) and len(car2.points), "Could not load car1.ply / car2.ply"

# 2) Downsample & normals (cars are larger → slightly bigger voxel)
v = 0.08
c1d = car1.voxel_down_sample(v);  c2d = car2.voxel_down_sample(v)
c1d.estimate_normals();           c2d.estimate_normals()

# 3) FPFH features
c1f = o3d.pipelines.registration.compute_fpfh_feature(
    c1d, o3d.geometry.KDTreeSearchParamHybrid(radius=v*5, max_nn=100))
c2f = o3d.pipelines.registration.compute_fpfh_feature(
    c2d, o3d.geometry.KDTreeSearchParamHybrid(radius=v*5, max_nn=100))

# 4) Global registration (RANSAC)
car_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    c1d, c2d, c1f, c2f,
    True, v*1.5,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    4
)

# (Optional) refine with ICP (point-to-plane) on full-res clouds
car1.estimate_normals(); car2.estimate_normals()
car_icp = o3d.pipelines.registration.registration_icp(
    car1, car2, v, car_ransac.transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

# 5) Visualize (use ICP transform if you ran it, else car_ransac.transformation)
draw_registrations(car1, car2, car_icp.transformation, True)


In [4]:
import open3d as o3d

# Load
car1 = o3d.io.read_point_cloud("global_registration/car1.ply")
car2 = o3d.io.read_point_cloud("global_registration/car2.ply")

# (If you have a RANSAC transform from Task 1 Part 2, put it here)
T_init = np.eye(4)  # replace with car_ransac.transformation if available

# Estimate normals (required for point-to-plane)
car1.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.3, max_nn=30))
car2.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.3, max_nn=30))

# Local registration (ICP, point-to-plane)
icp = o3d.pipelines.registration.registration_icp(
    car1, car2, 0.08, T_init,
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

print("ICP fitness:", icp.fitness, " RMSE:", icp.inlier_rmse)
draw_registrations(car1, car2, icp.transformation, True)


ICP fitness: 0.0  RMSE: 0.0


In [5]:
import glob
import open3d as o3d
import numpy as np
import copy
import os

# --- Collect RGB & Depth files ---
color_files = sorted(glob.glob("car_challange/rgb/*.jpg"))
depth_files = sorted(glob.glob("car_challange/depth/*.png"))

# --- Safety check ---
if not color_files or not depth_files:
    raise FileNotFoundError("❌ No RGB or Depth images found. Check folder paths and file extensions!")

print(f"✅ Found {len(color_files)} color and {len(depth_files)} depth images")

# --- Optional: Use fewer frames for faster processing ---
# Example 1: Take every 10th frame
# color_files = color_files[::10]; depth_files = depth_files[::10]

# Example 2: Use only first 50 frames
# color_files = color_files[:50]; depth_files = depth_files[:50]

# --- Camera intrinsics ---
camera = o3d.camera.PinholeCameraIntrinsic(
    o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault
)

# --- Build first frame ---
c0 = o3d.io.read_image(color_files[0])
d0 = o3d.io.read_image(depth_files[0])
rgbd0 = o3d.geometry.RGBDImage.create_from_color_and_depth(
    c0, d0, convert_rgb_to_intensity=True
)

scene = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd0, camera)
scene.transform([[1, 0, 0, 0],
                 [0, -1, 0, 0],
                 [0, 0, -1, 0],
                 [0, 0, 0, 1]])

# --- ICP registration loop ---
threshold = 0.06
for i in range(1, len(color_files)):
    print(f"Processing frame {i+1}/{len(color_files)} ...")

    # Load color and depth images
    c = o3d.io.read_image(color_files[i])
    d = o3d.io.read_image(depth_files[i])
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        c, d, convert_rgb_to_intensity=True
    )

    # Create point cloud
    frame = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera)
    frame.transform([[1, 0, 0, 0],
                     [0, -1, 0, 0],
                     [0, 0, -1, 0],
                     [0, 0, 0, 1]])

    # Estimate normals for ICP
    scene.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30))
    frame.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30))

    # Point-to-plane ICP
    icp = o3d.pipelines.registration.registration_icp(
        frame, scene, threshold, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )

    # Apply transformation and merge
    frame.transform(icp.transformation)
    scene += frame

    # Downsample every 5 frames to reduce size
    if i % 5 == 0:
        scene = scene.voxel_down_sample(0.05)

# --- Visualize / Save result ---
o3d.visualization.draw_geometries([scene])
# Optional: save output
# o3d.io.write_point_cloud("car_reconstruction.ply", scene)


✅ Found 1722 color and 1729 depth images
Processing frame 2/1722 ...
Processing frame 3/1722 ...
Processing frame 4/1722 ...
Processing frame 5/1722 ...
Processing frame 6/1722 ...
Processing frame 7/1722 ...
Processing frame 8/1722 ...
Processing frame 9/1722 ...
Processing frame 10/1722 ...
Processing frame 11/1722 ...
Processing frame 12/1722 ...
Processing frame 13/1722 ...
Processing frame 14/1722 ...
Processing frame 15/1722 ...
Processing frame 16/1722 ...
Processing frame 17/1722 ...
Processing frame 18/1722 ...
Processing frame 19/1722 ...
Processing frame 20/1722 ...
Processing frame 21/1722 ...
Processing frame 22/1722 ...
Processing frame 23/1722 ...
Processing frame 24/1722 ...
Processing frame 25/1722 ...
Processing frame 26/1722 ...
Processing frame 27/1722 ...
Processing frame 28/1722 ...
Processing frame 29/1722 ...
Processing frame 30/1722 ...
Processing frame 31/1722 ...
Processing frame 32/1722 ...
Processing frame 33/1722 ...
Processing frame 34/1722 ...
Processing

KeyboardInterrupt: 